# Kaggriculture | Adaptive Farm Intelligence — Fieldbook Rolling Market

A compact, reproducible Kaggriculture submission package.

## Method

Fieldbook keeps its original physical route and worker plan. At each visible three-day boundary after day 12, it only re-prioritizes already planned SELL orders by visible price and market inventory pressure.

The notebook writes the required `submission.tar.gz`.

In [ ]:
# Self-contained competition package.
import base64
import hashlib
import io
import tarfile
from pathlib import Path

ARCHIVE_B64 = (
    '...'
)
EXPECTED_SHA256 = "3819a27ea6ec10dad188a56a3e2928724155bf9e633d6111f05ea1d09a6be8d4"
payload = base64.b64decode(ARCHIVE_B64)
assert hashlib.sha256(payload).hexdigest() == EXPECTED_SHA256
with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
    assert archive.getnames() == ["main.py"]
    source = archive.extractfile("main.py").read()
    compile(source, "main.py", "exec")
Path("submission.tar.gz").write_bytes(payload)
print("submission.tar.gz ready")
print("sha256:", EXPECTED_SHA256)
print("agent source:", len(source), "bytes")
